In [0]:
%pip install databricks-labs-dqx
dbutils.library.restartPython()

In [0]:
catalog="your_catalog"
schema="your_schema"
source_table = "raw_customer_revenue"
target_table = "customer_revenue_valid"
quarantine_table = "customer_revenue_quarantine"
incident_table = "dq_incidents"
job_name = "demo_dqx"

In [0]:
import yaml
from databricks.labs.dqx.engine import DQEngine

In [0]:
data = [
    (1, 30, True, "Cologne"),
    (2, 100, True, "Berlin"),
    (3, 50, False, "Munich"),
    (4, 75, True, "Hamburg"),
    (5, 120, True, "Frankfurt"),
    (6, -90, False, "Stuttgart"),
    (7, -60, True, "Dusseldorf"),
    (8, 80, False, "Dortmund"),
    (9, 110, True, "Essen"),
    (10, 150, True, "Leipzig"),
    (11, 40, False, "Bremen"),
    (12, 70, True, "Dresden"),
    (13, 95, True, "Hanover"),
    (14, 130, False, "Nuremberg"),
    (15, 85, True, "Duisburg"),
    (16, 55, False, "Bochum"),
    (17, 65, True, "Wuppertal"),
    (18, 105, True, "Bielefeld"),
    (19, 140, False, "Bonn"),
    (20, 1250, True, ""),
]
columns = ["id", "revenue", "is_active", "city"]

df= spark.createDataFrame(data, columns)

display(df)

id,revenue,is_active,city
1,30,true,Cologne
2,100,true,Berlin
3,50,false,Munich
4,75,true,Hamburg
5,120,true,Frankfurt
6,-90,false,Stuttgart
7,-60,true,Dusseldorf
8,80,false,Dortmund
9,110,true,Essen
10,150,true,Leipzig


In [0]:
checks_yaml="""
- check:
    arguments:
      col_names: [id,revenue,is_active]
    function: is_not_null
  criticality: error
- check:
    arguments:
      col_name: city
    function: is_not_null_and_not_empty
  criticality: error
- check:
    arguments:
      col_name: revenue
      limit: 0
    function: is_not_less_than
  criticality: error
  name: revenue_is_positive
- check:
    arguments:
      col_name: revenue
      min_limit: 0
      max_limit: 999
    function: is_in_range
- check:
    function: sql_expression
    arguments:
        expression: is_active is True
  criticality: error
  name: is_active_is_true
"""

In [0]:
checks = yaml.safe_load(checks_yaml)
# validate the checks
dq_engine = DQEngine(None)
status = dq_engine.validate_checks(checks)
print(status)
assert not status.has_errors

No errors found


In [0]:
valid_df, quarantined_df = dq_engine.apply_checks_by_metadata_and_split(df, checks)

print("Quarantine:")
display(quarantined_df)

print("Valid:")
display(valid_df)

Quarantine:


id,revenue,is_active,city,_errors,_warnings
3,50,false,Munich,"List(List(is_active_is_true, Value is not matching expression: is_active is True, null, null, sql_expression, 2025-05-03T11:22:48.931Z, Map()))",null
6,-90,false,Stuttgart,"List(List(revenue_is_positive, Value '-90' in Column 'revenue' is less than limit: 0, revenue, null, is_not_less_than, 2025-05-03T11:22:48.931Z, Map()), List(col_revenue_not_in_range, Value '-90' in Column 'revenue' not in range: [0, 999], revenue, null, is_in_range, 2025-05-03T11:22:48.931Z, Map()), List(is_active_is_true, Value is not matching expression: is_active is True, null, null, sql_expression, 2025-05-03T11:22:48.931Z, Map()))",null
7,-60,true,Dusseldorf,"List(List(revenue_is_positive, Value '-60' in Column 'revenue' is less than limit: 0, revenue, null, is_not_less_than, 2025-05-03T11:22:48.931Z, Map()), List(col_revenue_not_in_range, Value '-60' in Column 'revenue' not in range: [0, 999], revenue, null, is_in_range, 2025-05-03T11:22:48.931Z, Map()))",null
8,80,false,Dortmund,"List(List(is_active_is_true, Value is not matching expression: is_active is True, null, null, sql_expression, 2025-05-03T11:22:48.931Z, Map()))",null
11,40,false,Bremen,"List(List(is_active_is_true, Value is not matching expression: is_active is True, null, null, sql_expression, 2025-05-03T11:22:48.931Z, Map()))",null
14,130,false,Nuremberg,"List(List(is_active_is_true, Value is not matching expression: is_active is True, null, null, sql_expression, 2025-05-03T11:22:48.931Z, Map()))",null
16,55,false,Bochum,"List(List(is_active_is_true, Value is not matching expression: is_active is True, null, null, sql_expression, 2025-05-03T11:22:48.931Z, Map()))",null
19,140,false,Bonn,"List(List(is_active_is_true, Value is not matching expression: is_active is True, null, null, sql_expression, 2025-05-03T11:22:48.931Z, Map()))",null
20,1250,true,,"List(List(col_city_is_null_or_empty, Column 'city' value is null or empty, city, null, is_not_null_and_not_empty, 2025-05-03T11:22:48.931Z, Map()), List(col_revenue_not_in_range, Value '1250' in Column 'revenue' not in range: [0, 999], revenue, null, is_in_range, 2025-05-03T11:22:48.931Z, Map()))",null


Valid:


id,revenue,is_active,city
1,30,true,Cologne
2,100,true,Berlin
4,75,true,Hamburg
5,120,true,Frankfurt
9,110,true,Essen
10,150,true,Leipzig
12,70,true,Dresden
13,95,true,Hanover
15,85,true,Duisburg
17,65,true,Wuppertal


In [0]:
import pyspark.sql.functions as F

# extract errors
errors_df = quarantined_df.select(
    F.explode(F.col("_errors")).alias("result")
).select(F.expr("result.*"), F.lit("error").alias("level"))

# extract errors
warnings_df = quarantined_df.select(
    F.explode(F.col("_warnings")).alias("result")
).select(F.expr("result.*"), F.lit("warning").alias("criticality"))

incidents_df = errors_df.union(warnings_df).select(
    F.lit(source_table).alias("source_table"),
    F.lit(quarantine_table).alias("quarantine_table"),
    F.lit(job_name).alias("job_name"),"*"
    )

display(incidents_df)

source_table,quarantine_table,job_name,name,message,col_name,filter,function,run_time,user_metadata,level
raw_customer_revenue,customer_revenue_quarantine,demo_dqx,is_active_is_true,Value is not matching expression: is_active is True,null,null,sql_expression,2025-05-03T11:22:48.931Z,Map(),error
raw_customer_revenue,customer_revenue_quarantine,demo_dqx,revenue_is_positive,Value '-90' in Column 'revenue' is less than limit: 0,revenue,null,is_not_less_than,2025-05-03T11:22:48.931Z,Map(),error
raw_customer_revenue,customer_revenue_quarantine,demo_dqx,col_revenue_not_in_range,"Value '-90' in Column 'revenue' not in range: [0, 999]",revenue,null,is_in_range,2025-05-03T11:22:48.931Z,Map(),error
raw_customer_revenue,customer_revenue_quarantine,demo_dqx,is_active_is_true,Value is not matching expression: is_active is True,null,null,sql_expression,2025-05-03T11:22:48.931Z,Map(),error
raw_customer_revenue,customer_revenue_quarantine,demo_dqx,revenue_is_positive,Value '-60' in Column 'revenue' is less than limit: 0,revenue,null,is_not_less_than,2025-05-03T11:22:48.931Z,Map(),error
raw_customer_revenue,customer_revenue_quarantine,demo_dqx,col_revenue_not_in_range,"Value '-60' in Column 'revenue' not in range: [0, 999]",revenue,null,is_in_range,2025-05-03T11:22:48.931Z,Map(),error
raw_customer_revenue,customer_revenue_quarantine,demo_dqx,is_active_is_true,Value is not matching expression: is_active is True,null,null,sql_expression,2025-05-03T11:22:48.931Z,Map(),error
raw_customer_revenue,customer_revenue_quarantine,demo_dqx,is_active_is_true,Value is not matching expression: is_active is True,null,null,sql_expression,2025-05-03T11:22:48.931Z,Map(),error
raw_customer_revenue,customer_revenue_quarantine,demo_dqx,is_active_is_true,Value is not matching expression: is_active is True,null,null,sql_expression,2025-05-03T11:22:48.931Z,Map(),error
raw_customer_revenue,customer_revenue_quarantine,demo_dqx,is_active_is_true,Value is not matching expression: is_active is True,null,null,sql_expression,2025-05-03T11:22:48.931Z,Map(),error


In [0]:
# Store the data to their targets
# 'overwrite' for demo

# Store valid data to target table -> further processing
valid_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{schema}.{target_table}"
)

# Store invalid data to quarantine table -> manually check and clean
quarantined_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{catalog}.{schema}.{quarantine_table}")

# Store incidents to the incident table -> reporting + alerting
incidents_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{schema}.{incident_table}"
)